In [ ]:
from pathlib import Path

COLAB_CWD = Path("/content")
CWD = Path.cwd()
IN_COLAB = False


def is_colab():
    return CWD == COLAB_CWD


if is_colab():
    IN_COLAB = True
    !git clone "https://github.com/fbocchi/show-and-tell.git"
    %cd "show-and-tell"

In [ ]:
from models.loss import loss_that_ignores_padding

In [ ]:
if IN_COLAB:
    %cd "notebooks"
    CWD = Path.cwd()

print(CWD)

In [ ]:
import os
import pandas as pd


train_df = pd.read_csv(f"{CWD}/../flickr8k/captions_train.csv")

image_paths = train_df["image"].apply(lambda x: os.path.join(f"{CWD}/../flickr8k/Images", x)).to_numpy()
captions = train_df["caption"].apply(lambda x: f"[START] {x} [END]").to_numpy()

print(captions)

In [ ]:
import json
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

from utils.load_image import load_image


with open(f"{CWD}/../config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

vectorizer = TextVectorization.from_config(config["vec_config"])

vocab = vectorizer.get_vocabulary()

print(len(vocab))
print(vocab)

def create_sample(image_path, caption):
    image = load_image(image_path)
    in_caption, target_caption = split_caption_for_teacher_forcing(caption)
    in_caption = vectorizer(in_caption)
    target_caption = vectorizer(target_caption)
    return (image, in_caption), target_caption

def split_caption_for_teacher_forcing(caption):
    tokens = tf.strings.split(caption, sep=" ")

    tok_in_caption = tokens[:-1]
    tok_target_caption = tokens[1:]

    in_caption = tf.strings.reduce_join(
        tok_in_caption,
        separator=" "
    )

    target_caption = tf.strings.reduce_join(
        tok_target_caption,
        separator=" "
    )

    return in_caption, target_caption

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((image_paths, captions))
dataset = dataset.map(create_sample, num_parallel_calls=tf.data.AUTOTUNE)

dataset = dataset.shuffle(1000)
dataset = dataset.batch(64)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
import keras

model = keras.models.load_model(
    "../models/model.keras",
    custom_objects={
        "loss_that_ignores_padding": loss_that_ignores_padding
    }
)

model.fit(dataset, epochs=1)

model.save("../models/model_train.keras")
